In [ ]:
import numpy as np
import pandas as p
import matplotlib.pyplot as plt
from matplotlib import pyplot
from sklearn.impute import SimpleImputer

data= p.read_csv("shopping_trends_updated.csv")

numeric_cols = data.select_dtypes(include=np.number).columns
non_numeric_cols = data.select_dtypes(exclude=np.number).columns

imp_numeric = SimpleImputer(strategy='mean')
data[numeric_cols] = imp_numeric.fit_transform(data[numeric_cols])

imp_non_numeric = SimpleImputer(strategy='most_frequent')
data[non_numeric_cols] = imp_non_numeric.fit_transform(data[non_numeric_cols])

In [35]:
x = data.drop(['Discount Applied', 'Promo Code Used','Subscription Status'],axis=1)
y=data['Subscription Status']

In [36]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=101)

In [38]:
from sklearn.svm import SVC 
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier, StackingClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score

combined_trf = ColumnTransformer(
    transformers= [('ss', StandardScaler(), [17, 18]), 
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), 
     [0, 2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16])],
    remainder='passthrough')

pca = PCA(n_components=10)

knn = KNeighborsClassifier()
svc = SVC(probability=True)
rfc = RandomForestClassifier()
dtc=DecisionTreeClassifier()

voting = VotingClassifier(estimators=[('knn', knn), ('svc', svc), ('rfc', rfc),('dtc',dtc)],voting='soft')
gbc = GradientBoostingClassifier()

estimator = [
    ('voting', voting),
    ('gbc', gbc)
]

stack = StackingClassifier(estimator, n_jobs=-1, final_estimator= LogisticRegression(penalty='l2',solver='saga'))

In [39]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
pipe = Pipeline([
     ('step1', combined_trf),
     ('step2', pca),
     ('step3', stack)
     ]
)
param_grid = {
    "step3__voting__knn__n_neighbors": [5, 11],
    "step3__voting__svc__C": [0.1,0.4,1],
    "step3__voting__svc__kernel": ['poly', 'rbf'],
    "step3__voting__svc__degree": [2, 3],
    "step3__voting__rfc__n_estimators": [64, 128],
    "step3__voting__rfc__max_depth": [None, 8, 12],    
    "step3__voting__rfc__min_samples_split": [15, 20, 30],
    "step3__gbc__learning_rate": [0.2, 0.4, 0.6],
    "step3__gbc__max_features": [8, 12],
    "step3__gbc__n_estimators": [128, 256],
    "step3__gbc__min_samples_split": [15, 25, 30],
    "step3__gbc__max_depth": [None, 7, 10]
}

search = RandomizedSearchCV(pipe, param_grid, n_jobs=-1, n_iter=2, scoring= 'accuracy', cv=5)
search.fit(x_train, y_train)

ValueError: 
All the 10 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\Diya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\__init__.py", line 456, in _get_column_indices_for_bool_or_int
    idx = _safe_indexing(np.arange(n_columns), key)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Diya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\__init__.py", line 411, in _safe_indexing
    return _array_indexing(X, indices, indices_dtype, axis=axis)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Diya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\__init__.py", line 208, in